# reduce-op-mean-divide — ex1: implement mean reduction as sum-then-in-place-divide

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `reduce-op-mean-divide`. Running the final beacon cell reports progress against the `Distributed: reduce-op mean divide` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: reduce-op mean divide` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`reduce-op-mean-divide`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "reduce-op-mean-divide"
DD_SUBTOPIC = "Distributed: reduce-op mean divide"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## torch.distributed quick refresher

PyTorch's collective-communication library (`torch.distributed`, aliased `dist`) lets multiple processes coordinate over tensors. The standard workflow:

1. **Each rank** runs the same function, parameterized by `rank` and `world_size`. Rank 0 is conventionally the 'driver'.
2. **`dist.init_process_group(backend=...)`** establishes the rendezvous. Backends:
   - `'nccl'` — NVIDIA's GPU-to-GPU primitive. Used in ARENA's multi-GPU setup. Requires CUDA + one process per GPU.
   - `'gloo'` — CPU-friendly. What you'll use in these drills (Colab CPU runtimes have no real GPUs).
3. **Pin a device** per rank: `torch.device(f'cuda:{rank}')` so each process owns exactly one GPU.
4. **Collective ops** (`all_reduce`, `broadcast`, `send`, `recv`) operate in-place on tensors of identical shape across all ranks.
5. **`dist.destroy_process_group()`** tears down at the end.

**Two ways to launch multiple ranks:**
- `torch.multiprocessing.spawn(fn, args=(...), nprocs=world_size)` — what ARENA uses. Spawn requires the worker fn be importable (not defined in `__main__`/a notebook cell).
- `mp.get_context('fork').Process(target=fn, args=...)` — Linux-only but works with cell-defined fns. The drills use this in tests so the worker can stay in the cell.

**Two-rank trick.** Colab gives ~2 CPU cores, so `world_size=2` is the right scale: enough to exercise the protocol, cheap enough to finish in seconds.

### This drill's atom: 'mean' = sum then in-place divide
`dist.ReduceOp` enum has `SUM`, `PRODUCT`, `MAX`, `MIN`, but **no `MEAN`**. To average tensors across ranks portably:
```python
dist.all_reduce(tensor, op=dist.ReduceOp.SUM)
tensor /= world_size   # in-place division
```
The `/=` matters — it mutates the tensor in place so anyone holding a reference (e.g. `param.grad`) sees the new value. `tensor = tensor / world_size` rebinds the local name but leaves the upstream `param.grad` reference pointing at the unscaled sum.

ARENA's implementation:
```python
if op == 'mean':
    tensor /= world_size
```
This drill turns the divide into a standalone function so you can verify the in-place semantics.

### Exercise 1 — implement mean reduction as sum-then-in-place-divide

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `dist.all_reduce(SUM)` followed by `tensor /= world_size` (in-place) to mean-reduce across ranks, and verify the in-place step preserves an upstream `.grad` reference that points at the same storage.
> Keywords: ReduceOp.SUM, mean, in-place-divide, aliasing, gloo
> ```

**KCs targeted:** `sum-then-divide-for-mean`, `inplace-divide-preserves-references`

Implement `ex1_mean_reduce_worker(rank, world_size, port, out_queue)`. Each rank:

1. Inits `gloo` process group.
2. Builds `param = t.nn.Parameter(t.zeros(3))` so `param.grad` can be assigned. Then sets `param.grad = t.tensor([float(rank+1), float(rank+1), float(rank+1)])` — rank-dependent gradient.
3. Calls `ex1_mean_reduce_inplace(param.grad, world_size)` (your function — defined below).
4. Pushes `(rank, param.grad.tolist())` AND a boolean `param.grad is grad_ref_at_start` (where you captured `grad_ref_at_start = param.grad` BEFORE step 3) onto `out_queue` as `(rank, grad_vals, ref_preserved_bool)`.
5. Destroys process group.

Implement `ex1_mean_reduce_inplace(tensor, world_size)`:
1. `dist.all_reduce(tensor, op=dist.ReduceOp.SUM)`.
2. **In-place** divide by world_size: `tensor /= world_size` (or equivalently `tensor.div_(world_size)`). DO NOT use `tensor = tensor / world_size` — that rebinds the local name and breaks the reference invariant.
3. Return `None` (function mutates in-place).

The test asserts:
- All ranks end with the same mean grad.
- The `param.grad is grad_ref_at_start` check is `True` on every rank (proves the in-place semantics).

In [ ]:
import os, datetime
import torch as t
import torch.distributed as dist

def ex1_mean_reduce_inplace(tensor, world_size):
    """All-reduce with sum then divide IN PLACE by world_size."""
    raise NotImplementedError()

def ex1_mean_reduce_worker(rank, world_size, port, out_queue):
    """Build param.grad, mean-reduce, queue (grad_vals, ref_preserved)."""
    raise NotImplementedError()


def _test_ex1():
    import os as _os
    import datetime as _dt
    import torch.distributed as _dist
    import torch.multiprocessing as _mp

    def _dd_run_workers(worker_fn, world_size, port, *extra_args, timeout=30):
        """Spawn `world_size` fork-context procs, return list of exitcodes."""
        ctx = _mp.get_context('fork')
        procs = []
        for rank in range(world_size):
            p = ctx.Process(target=worker_fn, args=(rank, world_size, port, *extra_args))
            p.start()
            procs.append(p)
        for p in procs:
            p.join(timeout=timeout)
        codes = [p.exitcode for p in procs]
        for p in procs:
            if p.is_alive():
                p.terminate()
        return codes

    manager = _mp.Manager()
    q = manager.Queue()
    codes = _dd_run_workers(ex1_mean_reduce_worker, 3, 29560, q)
    assert codes == [0, 0, 0], f'workers failed: {codes}'

    results = {}
    while not q.empty():
        rank, grad_vals, ref_preserved = q.get()
        results[rank] = (grad_vals, ref_preserved)

    # Pre-grads: rank 0 = [1,1,1], rank 1 = [2,2,2], rank 2 = [3,3,3]. Mean = [2,2,2].
    expected_grad = [2.0, 2.0, 2.0]
    for rank in [0, 1, 2]:
        grad_vals, ref_preserved = results[rank]
        for i, (a, b) in enumerate(zip(grad_vals, expected_grad)):
            assert abs(a - b) < 1e-5, f'rank {rank} idx {i}: got {a}, expected {b}'
        assert ref_preserved is True, (
            f'rank {rank}: param.grad reference was rebound — '
            f'did you write `tensor = tensor / world_size` instead of `tensor /= world_size`?'
        )

    # 2-rank case: [1,1,1] and [2,2,2] → [1.5, 1.5, 1.5].
    q2 = manager.Queue()
    codes2 = _dd_run_workers(ex1_mean_reduce_worker, 2, 29561, q2)
    assert codes2 == [0, 0]
    while not q2.empty():
        rank, gv, rp = q2.get()
        for v in gv:
            assert abs(v - 1.5) < 1e-5, f'2-rank: rank {rank} got {gv}'
        assert rp is True
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_mean_reduce_inplace(tensor, world_size):
    dist.all_reduce(tensor, op=dist.ReduceOp.SUM)
    tensor /= world_size  # in-place — preserves caller's reference

def ex1_mean_reduce_worker(rank, world_size, port, out_queue):
    os.environ['MASTER_ADDR'] = '127.0.0.1'
    os.environ['MASTER_PORT'] = str(port)
    dist.init_process_group(backend='gloo', rank=rank, world_size=world_size,
                            timeout=datetime.timedelta(seconds=20))
    param = t.nn.Parameter(t.zeros(3))
    param.grad = t.tensor([float(rank + 1)] * 3)
    grad_ref_at_start = param.grad
    ex1_mean_reduce_inplace(param.grad, world_size)
    ref_preserved = param.grad is grad_ref_at_start
    out_queue.put((rank, param.grad.tolist(), ref_preserved))
    dist.destroy_process_group()
```

**`/=` vs `/`.** In Python, `x /= y` calls `x.__itruediv__(y)` which (for `Tensor`) mutates in place. `x = x / y` calls `x.__truediv__(y)`, allocates a new tensor, and rebinds `x` to it.

For a free-standing local variable, both produce the right *value* but different *storage*. For a parameter's `.grad`, only the in-place form keeps the optimizer's grad reference in sync.

**Why no `ReduceOp.AVG` on gloo.** NCCL added `AVG` in PyTorch 1.10. Gloo never did. The portable, framework-agnostic recipe is always 'sum then divide'. If you only ever ship NCCL, `dist.all_reduce(t, op=dist.ReduceOp.AVG)` is cleaner — but the drill is gloo, so we use the portable form.

**Float-precision footgun.** Summing then dividing accumulates rounding error proportional to `world_size`. For most ML use cases this is negligible (`world_size < 1024` in practice). If you ever do see drift between ranks, suspect non-determinism in the reduction order, not the divide.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()